<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/extended/01_tabular_xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 1 · Hands-on: Explainability for tabular models

## Introduction

In this notebook we put into practice the **model-agnostic and model-specific explainability (XAI)** techniques seen in the first session, on a tabular regression problem. As models become more complex (tree ensembles, neural networks...), it becomes harder to understand *why* they make a given prediction.

We will use the **California housing** dataset and a gradient-boosted tree model (**XGBoost**), and tackle the problem from several angles:

1. **Transparent (glass-box) models**: a standardized linear model as a reference.
2. **Global feature importance**: which variables matter most? (model-based *gain* vs *permutation* importance)
3. **Partial Dependence (PDP) and ICE plots**: *how* does the prediction change with a feature? Including 2D interactions and the correlated-features pitfall.
4. **LIME**: local surrogate models for a single prediction.
5. **SHAP**: Shapley values, TreeSHAP vs KernelSHAP, local and global plots.
6. **Counterfactual explanations** with DiCE: what should change to get a different prediction?

**How to use this notebook.** Run the cells in order. Cells marked **Exercise** only require you to **change a value** in an existing code cell (look for the comment `# change me`), re-run it, and answer the question. Answers are hidden in collapsible cells: try first, then check! The whole notebook runs in a couple of minutes on a laptop CPU.

## Environment setup

On **Google Colab**, the next cell installs the missing libraries. On **your own computer**, install the requirements once (from the repository root) with:

```bash
pip install -r requirements.txt
```

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    %pip install -q shap xgboost lime dice-ml
else:
    print("Running locally: if any import below fails, run `pip install -r requirements.txt`")

In [ ]:
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import sklearn
import xgboost
import shap
import lime
import lime.lime_tabular
import dice_ml

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.neighbors import NearestNeighbors

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Plot style and some cosmetic settings
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)  # matplotlib deprecations triggered inside shap
warnings.filterwarnings('ignore', category=FutureWarning)

print(f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | shap {shap.__version__}")

## Data loading and model training

We use the **California Housing** dataset (1990 census): each row is a *block group* (a small district) and the goal is to predict the **median house value** of the district, in units of **$100,000** (the target is capped at 5.0, i.e. $500k).

| Feature | Meaning |
|---|---|
| `MedInc` | Median income in the district (tens of thousands of $) |
| `HouseAge` | Median house age (years) |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | District population |
| `AveOccup` | Average number of household members |
| `Latitude`, `Longitude` | Location of the district |

In [ ]:
# Downloads the dataset automatically (cached after the first run)
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
feature_names = list(X.columns)

print("Shape:", X.shape)
X.head()

In [ ]:
y.describe()

We split the data into training and test sets.

In [ ]:
# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

We train an **XGBoost regressor**: a powerful model, but a "black box" compared to a linear regression (it is a sum of 200 trees of depth 6).

In [ ]:
model = xgboost.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6,
                             random_state=SEED, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"XGBoost  -> test MSE: {mean_squared_error(y_test, y_pred):.4f} | test R2: {r2_score(y_test, y_pred):.4f}")

# Convenience prediction function that works with numpy arrays (needed by LIME / KernelSHAP)
def predict_fn(X_array):
    return model.predict(pd.DataFrame(X_array, columns=feature_names))

### A glass-box reference: standardized linear regression

Before explaining the black box, let us fit a **transparent model**. If we standardize the features (zero mean, unit variance), the coefficient of each feature is directly interpretable: *the change in predicted value (in $100k) when that feature increases by one standard deviation, keeping all the others fixed*.

In [ ]:
DROP_FEATURE = None  # change me: None, 'AveBedrms', 'Longitude', ...

lin_features = [f for f in feature_names if f != DROP_FEATURE]
linear = make_pipeline(StandardScaler(), LinearRegression())
linear.fit(X_train[lin_features], y_train)
print(f"Linear   -> test R2: {linear.score(X_test[lin_features], y_test):.4f}  (dropped: {DROP_FEATURE})")

coefs = pd.Series(linear[-1].coef_, index=lin_features).sort_values()
ax = coefs.plot.barh(figsize=(8, 4), color=np.where(coefs > 0, 'tab:red', 'tab:blue'))
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Coefficient (change in $100k per +1 std)')
ax.set_title('Standardized linear regression coefficients')
plt.show()

The linear model is fully transparent, but it is clearly less accurate (R² ≈ 0.58 vs ≈ 0.84 for XGBoost): this is the classic **accuracy / interpretability trade-off**. Notice also some puzzling signs: more rooms per household *decreases* the value, while more bedrooms *increases* it?

### Exercise 1.1 — Are linear coefficients really "the effect" of a feature?

`AveRooms` and `AveBedrms` are strongly correlated (ρ ≈ 0.85), and so are `Latitude` and `Longitude` (ρ ≈ −0.92). In the cell above, set `DROP_FEATURE = 'AveBedrms'` and re-run; then try `DROP_FEATURE = 'Longitude'`.

1. What happens to the coefficient of `AveRooms` (resp. `Latitude`)? Does the test R² change much?
2. What does this tell you about interpreting coefficients of a "transparent" model?

**Hint:** Each coefficient is the effect *keeping all the other features fixed*. What does "keeping `AveBedrms` fixed while increasing `AveRooms`" mean in practice?

<details><summary>Answer</summary>

1. Dropping `AveBedrms`, the `AveRooms` coefficient **flips sign** (≈ −0.29 → ≈ +0.05), and dropping `Longitude`, the `Latitude` coefficient goes from ≈ −0.90 to ≈ −0.06, while the R² barely changes (except when dropping `Longitude`, which carries information that `Latitude` alone cannot provide).
2. With correlated features, coefficients are *conditional* effects and they are unstable: "more rooms at a fixed number of bedrooms" means *more non-bedroom rooms* (bigger, more luxurious homes); "more bedrooms at a fixed number of rooms" is the opposite. Being transparent (we can read the formula) does not mean being easy to interpret, nor causal. The same issue (correlated features) will haunt every method in this notebook.

</details>

---

## 2. Global feature importance

The most basic form of global explainability is to determine which features contribute most to the model.

### Model-based importance (XGBoost)

Tree ensembles compute importance from the trees themselves:
- **weight**: number of times a feature is used to split.
- **gain**: average improvement of the loss brought by the splits on that feature.
- **cover**: average number of training samples affected by those splits.

In [ ]:
IMPORTANCE_TYPE = 'gain'  # change me: 'weight', 'gain' or 'cover'

booster_imp = pd.Series(model.get_booster().get_score(importance_type=IMPORTANCE_TYPE))
booster_imp = booster_imp.reindex(feature_names).fillna(0)
booster_imp = booster_imp / booster_imp.sum()  # normalize to sum 1

ax = booster_imp.sort_values().plot.barh(figsize=(8, 4), color='tab:gray')
ax.set_xlabel(f'Normalized importance ({IMPORTANCE_TYPE})')
ax.set_title(f'XGBoost built-in importance: {IMPORTANCE_TYPE}')
plt.show()

### Permutation importance (model-agnostic)

This method measures how much the model's score degrades (here, the drop in R²) if we "break" the relationship between a feature and the target by randomly **permuting** its values. It works with any model and is computed on held-out data.

In [ ]:
PERM_ON = 'test'  # change me: 'test' or 'train'

X_perm, y_perm = (X_test, y_test) if PERM_ON == 'test' else (X_train, y_train)
perm = permutation_importance(model, X_perm, y_perm, n_repeats=10, random_state=SEED, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feature_names)

perm_std = pd.Series(perm.importances_std, index=feature_names)
order = perm_imp.sort_values().index
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(order, perm_imp[order], xerr=perm_std[order], color='tab:green', capsize=3)
ax.set_xlabel('Decrease in R² when the feature is permuted')
ax.set_title(f'Permutation importance (on {PERM_ON} set)')
plt.show()

In [ ]:
# Side-by-side ranking (1 = most important)
pd.DataFrame({f'XGBoost {IMPORTANCE_TYPE}': booster_imp.rank(ascending=False).astype(int),
              f'Permutation ({PERM_ON})': perm_imp.rank(ascending=False).astype(int)}).sort_values(f'Permutation ({PERM_ON})')

**Why do they differ?** They answer different questions:
- **Gain** is computed at *training* time from the tree structure: it tells how much each feature was used to reduce the training loss. It is cheap, but specific to trees, measured in-sample, and biased toward features with many possible split points. Two features that are good substitutes may "share" or "steal" gain from each other.
- **Permutation importance** is model-agnostic and measured on held-out data in terms of the metric we care about. But permuting a feature on its own breaks its correlation with the other features, so the model is evaluated on **unrealistic inputs** (e.g., a district with the latitude of San Francisco and the longitude of Los Angeles). Part of the drop for `Latitude`/`Longitude` is the model extrapolating in regions where it has never seen data.

### Exercise 2.1 — Weight vs gain

Change `IMPORTANCE_TYPE` to `'weight'` and then to `'cover'`. Which features move up or down in the ranking compared to `'gain'`? Which one would you trust?

**Hint:** A feature can be used in *many* splits that each improve the loss only a little.

<details><summary>Answer</summary>

With `'gain'`, `MedInc` dominates (≈ 50% of the total), followed by `AveOccup`. With `'weight'`, `Latitude` and `Longitude` become the top features and the ranking is much flatter: they are continuous features with fine-grained structure, so the trees split on them *many* times (each split adds a small gain). Built-in importances depend on the arbitrary definition you choose; `gain` is usually the most meaningful of the three, but none of them measures the impact on *test* performance, which is what permutation importance does.

</details>

### Exercise 2.2 — Permutation importance on train vs test

Set `PERM_ON = 'train'` and re-run the permutation-importance cell (and the ranking cell). Compare the values with the test-set version. Which features gain importance on the training set, relative to their test values?

**Hint:** Look especially at the least important features (`AveBedrms`, `Population`). What would it mean for a feature to matter on training data but not on unseen data?

<details><summary>Answer</summary>

The values are somewhat larger on the training set, and the *relative* increase is largest for weak features: `AveBedrms` (≈ 0.007 → ≈ 0.026) and `Population` (≈ 0.005 → ≈ 0.023) are ~4× more "important" on train. The model has used these features to fit noise in the training set (overfitting), and that reliance does not generalize. **Compute permutation importance on held-out data** if you want to know what the model needs to *generalize*.

</details>

---

## 3. Partial Dependence (PDP) and ICE plots

While importance tells us *which* features matter, dependence plots tell us *how* they affect the prediction (positive, negative, linear, saturating...).

### 1D Partial Dependence Plots

The PDP of a feature $x_S$ is the average prediction when $x_S$ is set to a value $v$ for **every** instance: $\mathrm{PD}(v) = \frac{1}{n}\sum_i f(v, x_C^{(i)})$. We use a subsample of 1000 training points to keep it fast.

In [ ]:
X_pdp = X_train.sample(1000, random_state=SEED)  # subsample to keep PDP/ICE fast

features_to_plot = ['MedInc', 'AveOccup', 'HouseAge', 'Latitude', 'Longitude', 'AveRooms']

fig, ax = plt.subplots(figsize=(14, 7))
PartialDependenceDisplay.from_estimator(model, X_pdp, features_to_plot, kind='average', n_cols=3, ax=ax)
plt.suptitle('Partial Dependence Plots (PDP)', y=1.02)
plt.tight_layout()
plt.show()

### 2D Partial Dependence Plots (interactions)

We can visualize how two features jointly affect the prediction.

In [ ]:
features_2d = [
    ('Longitude', 'Latitude'),  # geographic interaction
    ('MedInc', 'AveOccup')      # socioeconomic interaction
]

fig, ax = plt.subplots(figsize=(14, 6))
PartialDependenceDisplay.from_estimator(model, X_pdp, features_2d, kind='average',
                                        grid_resolution=25, n_cols=2, ax=ax)
plt.suptitle('2D Partial Dependence Plots (interactions)', y=1.02)
plt.tight_layout()
plt.show()

### Individual Conditional Expectation (ICE)

The PDP is an average. ICE plots draw one curve **per instance** (the PDP is the mean of the ICE curves). If the curves are not parallel, the effect of the feature depends on the other features: there are **interactions** that the PDP hides. *Centered* ICE (c-ICE) anchors all curves at 0 on the left, which makes it easier to compare their shapes.

In [ ]:
ICE_FEATURE = 'MedInc'  # change me: 'MedInc', 'AveOccup', 'HouseAge', 'Latitude', ...
CENTERED = False        # change me: True for centered ICE (c-ICE)

fig, ax = plt.subplots(figsize=(8, 6))
PartialDependenceDisplay.from_estimator(
    model, X_pdp, [ICE_FEATURE],
    kind='both',       # individual ICE lines + their average (PDP)
    subsample=100,     # draw only 100 ICE curves to avoid clutter
    centered=CENTERED, random_state=SEED, ax=ax)
ax.set_title(f'ICE plot for {ICE_FEATURE} (centered={CENTERED})')
plt.show()

### The correlated-features pitfall

To compute the PDP of `Latitude`, every instance gets its latitude replaced by every grid value **while keeping its own longitude**. When features are correlated, many of these synthetic points do not exist in reality (e.g., a district in the Pacific Ocean or in the Nevada desert), and the model is evaluated where it has never seen data. Let us visualize the points that the PDP actually evaluates.

In [ ]:
PITFALL_PAIR = ('Longitude', 'Latitude')  # change me: ('AveRooms', 'AveBedrms'); the PDP varies the 2nd feature

a, b = PITFALL_PAIR
rows = X_pdp.sample(200, random_state=SEED)
grid = np.linspace(X[b].quantile(0.05), X[b].quantile(0.95), 20)  # the PDP grid for feature b
synthetic = pd.concat([rows.assign(**{b: v}) for v in grid])      # what the PDP evaluates

# A synthetic point is "unrealistic" if it is farther from any real district than
# 99% of real districts are from their nearest neighbour (in standardized units)
mu, sd = X[[a, b]].mean(), X[[a, b]].std()
nn = NearestNeighbors(n_neighbors=2).fit(((X[[a, b]] - mu) / sd).values)
threshold = np.quantile(nn.kneighbors()[0][:, 0], 0.99)
dist = nn.kneighbors(((synthetic[[a, b]] - mu) / sd).values, n_neighbors=1)[0][:, 0]
off = dist > threshold

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X[a], X[b], s=2, c='lightgray', label='real data')
ax.scatter(synthetic[a][~off], synthetic[b][~off], s=6, c='tab:blue', label='PDP point, realistic')
ax.scatter(synthetic[a][off], synthetic[b][off], s=6, c='tab:red', label='PDP point, unrealistic')
ax.set_xlim(X[a].quantile(0.001), X[a].quantile(0.999)); ax.set_ylim(X[b].quantile(0.001), X[b].quantile(0.999))
ax.set_xlabel(a); ax.set_ylabel(b); ax.legend(loc='best')
ax.set_title(f'Points evaluated by the PDP of {b}: {off.mean():.0%} unrealistic')
plt.show()

More than half of the points used to compute the `Latitude` PDP are far from any real district! Remedies include **ALE plots** (Accumulated Local Effects, which only use local, realistic changes), conditional sampling, or simply checking the data distribution (the rug/deciles in the PDP) before trusting the curve. Keep this in mind: permutation importance and *interventional* SHAP also evaluate the model on such synthetic points.

### Exercise 3.1 — Heterogeneous effects with ICE

In the ICE cell, set `CENTERED = True` and try `ICE_FEATURE = 'MedInc'`, `'AveOccup'`, `'Latitude'` and `'HouseAge'`. For which feature are the ICE curves most heterogeneous (non-parallel)? What does that imply about the PDP of that feature?

**Hint:** With centered curves, if all lines overlap, the feature acts additively (same effect for everyone). Large fan-outs mean the effect depends on the values of other features.

<details><summary>Answer</summary>

All of them show heterogeneity (non-parallel curves), i.e. **interactions**:
- `MedInc`: all curves increase, but with very different amplitudes (the fan opens up as income grows).
- `AveOccup`: the drop ranges from almost 0 to more than −2 depending on the district.
- `Latitude` is the most heterogeneous: for some districts the prediction drops by 4+ (in $100k) when moving north, for others it is almost flat, because latitude strongly interacts with longitude (moving north along the coast is not the same as moving north inland).
- `HouseAge` is the most instructive: the PDP is almost **flat**, suggesting the feature does not matter, but the ICE curves go **up for some districts and down for others**, and these opposite effects cancel out in the average.

When ICE curves are heterogeneous, the PDP can describe *no actual instance* well.

</details>

### Exercise 3.2 — Is the pitfall always severe?

Set `PITFALL_PAIR = ('AveRooms', 'AveBedrms')` and re-run. Compare the fraction of unrealistic points with the `('Longitude', 'Latitude')` case. Then try `('Latitude', 'Longitude')`.

**Hint:** Correlation coefficients alone do not tell the whole story: look at the *shape* of the joint distribution of each pair.

<details><summary>Answer</summary>

For `AveRooms`/`AveBedrms` only ≈ 2% of the points are flagged, while for `Latitude`/`Longitude` more than 50% are. Although `AveRooms` and `AveBedrms` are correlated (ρ ≈ 0.85), their joint distribution is a fairly wide cloud within the 5–95% PDP grid, so most combinations remain plausible. The geographic pair instead lives on a thin, curved manifold (the shape of California), so almost any change of one coordinate alone falls outside it. The severity of the pitfall depends on how concentrated the joint distribution is, not only on the correlation coefficient.

</details>

---

## 4. LIME (Local Interpretable Model-agnostic Explanations)

LIME explains **one prediction** by fitting a simple, interpretable **surrogate model** (a weighted linear regression) around the instance:
1. Sample perturbed points around the instance (for tabular data, from the training distribution; continuous features are discretized into quartile bins by default).
2. Get the black-box predictions for these samples.
3. Weight the samples by their proximity to the instance using an exponential kernel of width `kernel_width`.
4. Fit a weighted linear model; its coefficients are the explanation.

The resulting weights refer to *conditions* such as `MedInc <= 2.57` (the bin the instance falls in), so they read as "being in this bin moves the prediction by w".

In [ ]:
LIME_IDX = 0          # change me: instance of the test set to explain
KERNEL_WIDTH = None   # change me: None (default = 0.75*sqrt(n_features) ≈ 2.1), 0.5, 1, 5...
NUM_SAMPLES = 5000    # change me: number of perturbed samples (e.g. 500)
LIME_SEED = 0         # change me: random seed of the sampler

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, feature_names=feature_names, mode='regression',
    discretize_continuous=True, kernel_width=KERNEL_WIDTH, random_state=LIME_SEED)

x_lime = X_test.iloc[LIME_IDX]
lime_exp = lime_explainer.explain_instance(x_lime.values, predict_fn, num_features=len(feature_names),
                                           num_samples=NUM_SAMPLES)

print(f"Model prediction:          {lime_exp.predicted_value:.3f}")
print(f"Surrogate local prediction:{lime_exp.local_pred[0]:.3f}")
print(f"Surrogate fidelity (weighted R² on the perturbed samples): {lime_exp.score:.3f}")
lime_exp.as_pyplot_figure()
plt.title(f'LIME explanation for test instance {LIME_IDX}')
plt.show()

### LIME vs SHAP for the same instance

Let us compare with the SHAP values for the same instance (we will study SHAP in detail in the next section). We use TreeSHAP with a background sample of 100 training points.

In [ ]:
background = shap.sample(X_train, 100, random_state=SEED)  # background (reference) data, reused in Section 5
shap_explainer = shap.TreeExplainer(model, data=background)

shap_local = pd.Series(shap_explainer(X_test.iloc[[LIME_IDX]]).values[0], index=feature_names)
lime_weights = pd.Series({feature_names[i]: w for i, w in lime_exp.as_map()[1]})
lime_rules = {feature_names[i]: rule for (rule, _), (i, _) in zip(lime_exp.as_list(), lime_exp.as_map()[1])}

order = shap_local.abs().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, vals, title, labels in [(axes[0], lime_weights[order], 'LIME weights', [lime_rules[f] for f in order]),
                                (axes[1], shap_local[order], 'SHAP values', [f'{f} = {x_lime[f]:.2f}' for f in order])]:
    ax.barh(labels, vals, color=np.where(vals > 0, 'tab:red', 'tab:blue'))
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(title)
plt.suptitle(f'Test instance {LIME_IDX}: prediction = {lime_exp.predicted_value:.2f}', y=1.02)
plt.tight_layout()
plt.show()

Both methods agree on the main drivers (`Latitude`, `MedInc`, `AveOccup`, `AveRooms`) and their sign, but not on everything (look at `Longitude`), and the numbers mean different things: SHAP values add up exactly to *prediction − average prediction*, while LIME weights are coefficients of a local surrogate that may fit the black box poorly. Check the outputs of the LIME cell: the fidelity score is low and the surrogate's local prediction (≈ 1.39) is far from the model's actual prediction (≈ 0.57)!

### Exercise 4.1 — How stable is LIME?

Re-run the LIME cell with `LIME_SEED = 1` and `LIME_SEED = 2`. Then set `NUM_SAMPLES = 500` and repeat with a few seeds. Do the weights change? Does the ranking of the features change?

**Hint:** LIME is based on random sampling: the explanation is a random variable. Compare to SHAP, which for trees is computed exactly.

<details><summary>Answer</summary>

With 5000 samples the weights change by a few hundredths between seeds and the ranking is mostly stable. With 500 samples the explanation becomes noticeably unstable: the 3rd–4th most important features change between seeds (e.g., `AveRooms` vs `AveOccup`) and even the top-2 order can swap. Always check LIME's stability (several seeds, enough samples) before trusting it; TreeSHAP is deterministic.

</details>

### Exercise 4.2 — The kernel width defines "local"

Try `KERNEL_WIDTH = 0.5` and then `KERNEL_WIDTH = 5` (with `NUM_SAMPLES = 5000`, `LIME_SEED = 0`). How do the weights, the surrogate local prediction and the fidelity score change?

**Hint:** A small kernel width gives weight only to the very closest samples; a very large width weights all samples equally, so the surrogate becomes a *global* linear model of the black box.

<details><summary>Answer</summary>

With `0.5` the explanation becomes much more local: the weights shrink, the ranking changes (e.g., `Longitude` drops out of the top features), the fidelity score increases (≈ 0.13 → ≈ 0.27) and the surrogate's local prediction gets closer to the model's (≈ 1.39 → ≈ 1.0, vs 0.57), but fewer samples effectively count, so it is noisier across seeds. With `5` (or larger) the result barely changes with respect to the default, since all samples already get similar weights: the explanation is close to a global linear fit on the binarized features. The kernel width is a critical, hard-to-choose hyperparameter: **different widths give different explanations** for the same prediction, and nothing tells you which one is "right". Also note the low fidelity overall: a linear model on 8 binary features cannot reproduce the black box well even locally.

</details>

---

## 5. SHAP (SHapley Additive exPlanations)

SHAP assigns each feature a contribution $\phi_j$ based on **Shapley values** from cooperative game theory: the average marginal contribution of feature $j$ over all possible coalitions of features. The explanation is additive:

$$f(x) = \underbrace{E[f(X)]}_{\text{base value}} + \sum_{j=1}^{M} \phi_j(x)$$

The value of a coalition $S$ is the expected prediction when the features in $S$ are fixed to the values of $x$ and the rest are drawn from a **background dataset** (the *interventional* formulation). Computing it exactly requires $2^M$ coalitions, so we need efficient estimators:
- **KernelSHAP**: model-agnostic; samples coalitions and solves a weighted linear regression (a LIME with a special kernel). Cost ≈ `nsamples × |background|` model evaluations per instance.
- **TreeSHAP**: model-specific for tree ensembles; computes the *exact* Shapley values in polynomial time by traversing the trees.

### TreeSHAP vs KernelSHAP

First, TreeSHAP for the whole test set (4128 instances):

In [ ]:
t0 = time.perf_counter()
shap_values = shap_explainer(X_test)  # interventional TreeSHAP w.r.t. the 100-point background
t_tree_all = time.perf_counter() - t0
print(f"TreeSHAP: {len(X_test)} instances in {t_tree_all:.1f} s ({1000 * t_tree_all / len(X_test):.2f} ms/instance)")
print("shap_values.shape =", shap_values.shape)

Now KernelSHAP for a few instances, with the **same background**, so both estimate the same quantity.

In [ ]:
N_KERNEL = 20            # number of test instances to explain with KernelSHAP
KERNEL_NSAMPLES = 'auto' # change me: 'auto', 20, 50, 100 (number of coalitions sampled per instance)

X_small = X_test.iloc[:N_KERNEL]
kernel_explainer = shap.KernelExplainer(predict_fn, background)

np.random.seed(SEED)
t0 = time.perf_counter()
sv_kernel = kernel_explainer.shap_values(X_small, nsamples=KERNEL_NSAMPLES, silent=True)
t_kernel = time.perf_counter() - t0

t0 = time.perf_counter()
sv_tree = shap_explainer(X_small).values
t_tree = time.perf_counter() - t0

print(f"TreeSHAP:   {1000 * t_tree / N_KERNEL:7.2f} ms/instance")
print(f"KernelSHAP: {1000 * t_kernel / N_KERNEL:7.2f} ms/instance  (nsamples={KERNEL_NSAMPLES})")
print(f"Max |TreeSHAP - KernelSHAP| = {np.abs(sv_tree - sv_kernel).max():.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(sv_tree.ravel(), sv_kernel.ravel(), s=12, alpha=0.7)
lims = [min(sv_tree.min(), sv_kernel.min()), max(sv_tree.max(), sv_kernel.max())]
ax.plot(lims, lims, 'k--', lw=0.8)
ax.set_xlabel('TreeSHAP (exact)'); ax.set_ylabel('KernelSHAP (approx.)')
ax.set_title(f'SHAP values of {N_KERNEL} instances × {len(feature_names)} features')
plt.show()

With only $M=8$ features there are just $2^8 = 256$ coalitions, so with `nsamples='auto'` KernelSHAP can enumerate all of them and matches TreeSHAP almost exactly. And because the model is very fast to evaluate, the runtime is in the same ballpark here. But KernelSHAP's cost grows with the number of features (exponentially if exact, and many more samples are needed for a good approximation) and with the size of the background, and each call to an expensive model (e.g., a deep network) multiplies it: for larger problems TreeSHAP (or other model-specific estimators) is orders of magnitude faster, and *exact*.

> Note: `shap.TreeExplainer(model)` *without* background data uses a different, faster variant (`feature_perturbation='tree_path_dependent'`) that approximates the conditional expectation using the training cover of the tree nodes; its values can differ slightly from the interventional ones used here.

### Exercise 5.1 — KernelSHAP is an approximation

Set `KERNEL_NSAMPLES = 20`, then `50` and `100`, and re-run. How does the max difference with TreeSHAP and the scatter plot change? How does the runtime change?

**Hint:** Below 256 coalitions, KernelSHAP has to *sample* coalitions and solve a regularized regression.

<details><summary>Answer</summary>

With 20 coalitions the estimates are clearly off (max error ≈ 0.36, i.e. $36k); the error roughly halves each time `nsamples` doubles (≈ 0.16 with 50, ≈ 0.07 with 100), and becomes ≈ 0 with `'auto'` (all 256 coalitions enumerated). Runtime decreases with fewer samples. In high-dimensional problems we can never enumerate all coalitions, so KernelSHAP is always a noisy estimate: check its convergence (vary `nsamples`), as with LIME.

</details>

### Local explanations (individual predictions)

Since SHAP values are additive, the base value plus the sum of the SHAP values of each instance must recover the model prediction (*local accuracy* / *efficiency*):

In [ ]:
reconstructed = shap_values.base_values + shap_values.values.sum(axis=1)
print("Base value E[f(X)] =", round(float(shap_values.base_values[0]), 3))
print("Local accuracy holds:", np.allclose(reconstructed, y_pred, atol=1e-3))

# Indices of the test instances with the largest and smallest absolute errors (useful for the exercise below)
abs_err = np.abs(y_test.values - y_pred)
idx_worst, idx_best = int(abs_err.argmax()), int(abs_err.argmin())
print(f"idx_worst = {idx_worst}, idx_best = {idx_best}")

#### Waterfall plot

Shows how, starting from the base value (the average prediction over the background), each feature pushes the prediction up (red) or down (blue).

In [ ]:
IDX = 0  # change me: any test index, e.g. idx_worst or idx_best

print(f"Test instance {IDX}: true value = {y_test.iloc[IDX]:.2f}, predicted = {y_pred[IDX]:.2f} (x $100k)")
display(X_test.iloc[[IDX]])
shap.plots.waterfall(shap_values[IDX])

#### Force plot

A compact alternative view of the same explanation.

In [ ]:
shap.plots.force(shap_values[IDX].base_values, shap_values[IDX].values, X_test.iloc[IDX].round(2),
                 matplotlib=True, figsize=(18, 3.5), text_rotation=15)

### Exercise 5.2 — Debugging the worst prediction

Set `IDX = idx_worst` in the waterfall cell and re-run it (and the force plot). Where is this district? Which features push the prediction up? Looking at the feature values, why do you think the model fails?

**Hint:** Compare the feature values of this district with typical values (e.g., the `X.head()` table): look at `HouseAge`, `Population` and `AveOccup`.

<details><summary>Answer</summary>

It is a district in the Bay Area (Oakland/Berkeley: lat 37.82, long −122.29) with a true value of only $60k but a prediction of ≈ $390k. The `Latitude` and `Longitude` contributions roughly cancel out (≈ −0.85 and ≈ +0.87), and the prediction is pushed up mainly by `HouseAge` = 2 (+0.87: brand-new houses), `Population` = 94 (+0.42) and the low `AveOccup` ≈ 1.6. The district is very atypical: tiny, brand new, with very few people per household (e.g., a new development, or special/institutional housing for which census values are unreliable). In the training data, such combinations in the Bay Area are expensive, so the model extrapolates "new + few residents + Bay Area → expensive". SHAP does not fix the model, but it tells us *why* it failed, which is the first step for debugging (data quality, missing features, outliers...).

</details>

### Global explanations (model behaviour)

Aggregating the local SHAP values of many instances gives us global explanations.

#### Bar plot

The global importance of each feature as the mean absolute SHAP value.

In [ ]:
shap.plots.bar(shap_values)

#### Beeswarm plot (summary)

Combines the importance (magnitude) with the effect (sign and color) of each feature:
- **Y axis**: features sorted by importance.
- **X axis**: SHAP value (impact on the prediction).
- **Color**: feature value (red = high, blue = low).

In [ ]:
shap.plots.beeswarm(shap_values)

#### Dependence plot (scatter)

Shows the relationship between the value of a feature and its SHAP value (a "data-driven" PDP, but evaluated only at real instances). The vertical spread at a fixed feature value indicates interactions; SHAP colors the points by the feature that seems to interact the most, or by the one you choose.

In [ ]:
DEP_FEATURE = 'MedInc'  # change me: any feature, e.g. 'Latitude', 'AveOccup', 'HouseAge'
COLOR_FEATURE = None    # change me: None (automatic choice) or a feature name, e.g. 'Longitude'

color = shap_values if COLOR_FEATURE is None else shap_values[:, COLOR_FEATURE]
shap.plots.scatter(shap_values[:, DEP_FEATURE], color=color, alpha=0.3,
                   xmin=shap_values[:, DEP_FEATURE].percentile(0.5), xmax=shap_values[:, DEP_FEATURE].percentile(99.5))

### Exercise 5.3 — Reading the global plots

1. In the beeswarm, look at `MedInc`: when its value is high (red), is the SHAP value positive or negative? Is the relationship monotonic?
2. In the dependence cell, set `DEP_FEATURE = 'Latitude'` and `COLOR_FEATURE = 'Longitude'`. Is the effect of latitude the same for all districts? Then try `DEP_FEATURE = 'AveOccup'` with `COLOR_FEATURE = None`.

**Hint:** Vertical spread at a fixed x-value means the effect of the feature depends on other features. Compare the points of similar latitude but different color (longitude).

<details><summary>Answer</summary>

1. High `MedInc` (red) gives large positive SHAP values and low income gives negative ones: higher income → more expensive houses, in a monotonic, roughly linear and then saturating way (the target is capped at $500k).
2. The SHAP value of `Latitude` decreases overall when moving north, but for a given latitude it spreads vertically depending on longitude (e.g., around 37–38°, the westernmost Bay Area districts get higher values than the inland ones). The geographic effect is not separable: both features jointly encode "where in California", and SHAP splits this joint effect between them. For `AveOccup`, low occupancy has a positive effect and higher occupancy (crowded households, often poorer districts) a negative one, with a steep drop for small values (≈ 1.5–4) and a flat effect beyond.

</details>

### Geographic analysis with SHAP

The housing market depends heavily on location ("location, location, location"). Since SHAP values are additive, the **sum** of the `Latitude` and `Longitude` SHAP values gives the total contribution of the location for each district. Let us plot it on a map.

In [ ]:
shap_lat = shap_values[:, 'Latitude'].values
shap_lon = shap_values[:, 'Longitude'].values
location_effect = shap_lat + shap_lon

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, vals, title in zip(axes, [shap_lon, shap_lat, location_effect],
                           ['SHAP(Longitude)', 'SHAP(Latitude)', 'SHAP(Latitude) + SHAP(Longitude)']):
    sc = ax.scatter(X_test['Longitude'], X_test['Latitude'], c=vals, s=6, alpha=0.8,
                    cmap='RdBu_r', norm=TwoSlopeNorm(vcenter=0))
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_title(title)
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, label='SHAP value ($100k)', shrink=0.8)
plt.tight_layout()
plt.show()

The combined map clearly shows the expensive coastal areas (San Francisco Bay Area, Los Angeles coast) in red and the cheaper inland areas (Central Valley, inland southern California) in blue. Individually, each coordinate's map is misleading: `Longitude` gets large positive values in the north and `Latitude` in the south, and these largely cancel out. Because the two features are strongly correlated, the model splits the location effect between them in a fairly arbitrary way; only their sum is meaningful.

---

## 6. Counterfactual explanations with DiCE

A counterfactual explanation answers: **"what is the smallest change to the input that would change the prediction to a desired outcome?"** For example: *"the model would predict a value between $250k and $350k for this district if its median income were $79k instead of $47k"*. Good counterfactuals should be **valid** (they reach the desired outcome), **proximal** (small changes), **sparse** (few features changed), **diverse** and **actionable/plausible**.

We use [DiCE](https://github.com/interpretml/DiCE) (*Diverse Counterfactual Explanations*) in **regression mode**, asking for a prediction inside a `desired_range`. We use its model-agnostic `random` search method, which is fast.

In [ ]:
dice_data = dice_ml.Data(dataframe=X_train.assign(MedHouseVal=y_train),
                         continuous_features=feature_names, outcome_name='MedHouseVal')
dice_model = dice_ml.Model(model=model, backend='sklearn', model_type='regressor')
dice_explainer = dice_ml.Dice(dice_data, dice_model, method='random')

def show_cfs(cf_result, query):
    # Table with the query and its counterfactuals; '-' marks unchanged features
    cfs = cf_result.cf_examples_list[0].final_cfs_df
    if cfs is None or len(cfs) == 0:
        print("No counterfactuals found!")
        return None
    table = pd.concat([query[feature_names], cfs[feature_names].astype(float)], ignore_index=True)
    table['prediction'] = model.predict(table[feature_names])
    out = table.round(2).astype(object)
    unchanged = np.isclose(table[feature_names].values, table[feature_names].values[0], rtol=1e-4)
    unchanged[0] = False
    out.loc[:, feature_names] = out[feature_names].mask(unchanged, '-')
    out.index = ['query'] + [f'CF {i + 1}' for i in range(len(cfs))]
    return out

In [ ]:
CF_IDX = 5  # instance of the test set to explain
query = X_test.iloc[[CF_IDX]]
print(f"Test instance {CF_IDX}: predicted value = {model.predict(query)[0]:.2f} (x $100k)")
query

First, we let DiCE change **any** feature:

In [ ]:
DESIRED_RANGE = [2.5, 3.5]  # change me: target prediction range (x $100k)
TOTAL_CFS = 4

cf_all = dice_explainer.generate_counterfactuals(query, total_CFs=TOTAL_CFS, desired_range=DESIRED_RANGE,
                                                 random_seed=SEED)
show_cfs(cf_all, query)

Most counterfactuals increase `MedInc`, which makes sense, but some may also suggest **moving the district** (changing `Longitude`/`Latitude`) or absurd values (e.g., 80+ rooms per household): they are valid for the model, but not *actionable* or *plausible*. DiCE lets us constrain the search:
- `features_to_vary`: the only features that may change (e.g., location is immutable).
- `permitted_range`: allowed range for some features (e.g., realistic values of `AveRooms`).

In [ ]:
FEATURES_TO_VARY = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'AveOccup']  # change me
PERMITTED_RANGE = {'AveRooms': [3, 8], 'AveOccup': [1, 5]}                      # change me (or None)

cf_constrained = dice_explainer.generate_counterfactuals(
    query, total_CFs=TOTAL_CFS, desired_range=DESIRED_RANGE, random_seed=SEED,
    features_to_vary=FEATURES_TO_VARY, permitted_range=PERMITTED_RANGE)
show_cfs(cf_constrained, query)

### Exercise 6.1 — Actionability constraints

In the constrained cell, remove `MedInc` from `FEATURES_TO_VARY` (keep the location fixed too) and set `PERMITTED_RANGE = None`. Look at the counterfactuals. Then restore `PERMITTED_RANGE = {'AveRooms': [3, 8], 'AveOccup': [1, 5]}`. What happens?

**Hint:** Without income, which feature can the model "abuse" to increase the prediction? How many training districts have more than, say, 20 rooms per household?

<details><summary>Answer</summary>

Without `MedInc` and without ranges, DiCE still finds "valid" counterfactuals, but absurd ones: ≈ 100 rooms per household and ≈ 950 people per household! These lie in regions of the input space with (almost) no training data (only 69 of the 20640 districts have more than 20 rooms per household), where the model extrapolates arbitrarily. They are **adversarial-like** counterfactuals: valid for the model but useless (and misleading) for a human. With realistic ranges, DiCE needs to combine several moderate changes (very new or very old houses, more rooms, fewer occupants...), so the counterfactuals are less sparse but more plausible. Counterfactuals are only as good as the constraints (plausibility, actionability) we impose.

</details>

### Exercise 6.2 — More ambitious targets

Set `DESIRED_RANGE = [4, 5]` (a value of $400–500k) and re-run both counterfactual cells (unconstrained and constrained with the default settings). How do the changes compare with the `[2.5, 3.5]` target?

**Hint:** Look at the magnitude of the changes in `MedInc` and at which other features DiCE needs to change. Are the counterfactuals still close to the original instance?

<details><summary>Answer</summary>

The farther the target, the larger and less plausible the changes. Unconstrained, DiCE mostly **moves the district towards the coast** (changing `Longitude`), sometimes together with absurd `AveRooms` (40–70 rooms). With the actionability constraints, income must roughly double or triple (`MedInc` ≈ 9.6–14 vs 4.7). Proximity and plausibility degrade quickly for distant targets, and the diversity of the solutions shows that there is no single "explanation": counterfactuals are a set of possible scenarios, not the cause of the prediction.

</details>

---

## Summary

| Method | Scope | Model-agnostic? | Cost | Main caveats |
|---|---|---|---|---|
| Linear coefficients | Global | No (glass-box) | Free | Conditional on the other features; unstable with correlated features |
| Built-in importance (gain) | Global | No (trees) | Free | Training-time, depends on definition (weight/gain/cover) |
| Permutation importance | Global | Yes | Medium | Breaks correlations → unrealistic inputs; use held-out data |
| PDP / ICE | Global / local | Yes | Medium | PDP averages hide interactions (use ICE); extrapolation with correlated features |
| LIME | Local | Yes | Low–medium | Random (unstable); sensitive to kernel width; low fidelity |
| KernelSHAP | Local (+ global by aggregation) | Yes | High | Approximation; cost grows with features × background size |
| TreeSHAP | Local (+ global by aggregation) | No (trees) | Low | Exact, but *which* Shapley value (interventional vs path-dependent) matters |
| Counterfactuals (DiCE) | Local | Yes | Low–medium | Many valid solutions; need plausibility/actionability constraints |

In the next session we move on to explaining **deep learning** models.